# 01 — Fondamentaux Azure & PaaS

**Objectif :** poser les **bases Azure & IA** (style AZ-900 / AI-900), puis **déployer concrètement** deux services PaaS Azure depuis ce notebook avec l'`az cli` :

1. **App Service** — héberger une application web sans gérer de serveur.
2. **Microsoft Foundry** — la plateforme Azure pour construire des applications et des agents IA.

À la fin du notebook, vous aurez :
- compris les concepts cloud & Azure (§ 1.1) et les concepts IA (§ 1.2) ;
- compris ce qu'est Microsoft Foundry et sa hiérarchie (§ 2) ;
- créé ces ressources via la ligne de commande ;
- supprimé proprement vos ressources individuelles.

### Structure du notebook

| § | Contenu |
|---|---------|
| **1** | **Concepts fondamentaux** — théorie |
| 1.1 | Azure & le cloud — *style AZ-900* |
| 1.2 | IA — du ML à la GenAI — *style AI-900* |
| **2** | **Microsoft Foundry** — la plateforme IA d'Azure |
| 3 | Pré-requis : Azure CLI & SDK Python |
| 4 | Définir nos variables |
| 5 | Vérifier le Resource Group existant |
| 6 | App Service — déployer une Web App |
| 7 | Provisionner Microsoft Foundry |
| 8 | Vue d'ensemble |
| 9 | 🧹 Nettoyage |

> 💡 Exécutez **chaque cellule l'une après l'autre** et lisez les explications. Tout ce qui commence par `!` dans une cellule Python est exécuté dans le shell.
>
> 🤖 La suite logique est le **notebook 02 — Déployer un modèle + un Agent IA dans Foundry**, qui réutilise le projet créé ici.
>
> 📈 Le **monitoring** de l'application (Application Insights) + l'**évaluation** des agents sont traités dans le **notebook 06**.

## 1. Concepts fondamentaux — Azure & IA

Avant de passer au code, on prend le temps de poser les bases. Ce chapitre est volontairement théorique et inspiré des deux certifications Microsoft débutantes :

- **1.1 Azure & le cloud** — inspiré d'[**AZ-900** *Azure Fundamentals*](https://learn.microsoft.com/training/paths/microsoft-azure-fundamentals-describe-cloud-concepts/).
- **1.2 IA — du ML à la GenAI** — inspiré d'[**AI-900** *AI Fundamentals*](https://learn.microsoft.com/training/paths/get-started-with-artificial-intelligence-on-azure/).

Si vous connaissez déjà ces sujets, vous pouvez survoler — on revient sur les points-clés plus loin de manière concrète.

---

### 1.1 Azure & le cloud — *style AZ-900*

#### 1.1.1 Le cloud, c'est quoi ?

Le **cloud computing**, c'est consommer des ressources informatiques (calcul, stockage, réseau, IA…) **à la demande, via Internet**, facturées à l'usage. Plus besoin d'acheter ses serveurs.

**Pourquoi c'est intéressant ?**

| Bénéfice | Concrètement |
|----------|--------------|
| **Élasticité** (scale up / out) | Monter/descendre la capacité en quelques secondes. |
| **Haute disponibilité & fiabilité** | SLA jusqu'à 99,99 % grâce aux datacenters redondants. |
| **OpEx vs CapEx** | On paie à l'usage, pas en investissement initial. |
| **Couverture mondiale** | Plus de 60 régions Azure. |
| **Sécurité & conformité** | Certifications ISO, SOC, HDS, RGPD… intégrées. |
| **Time-to-market** | On déploie une appli en minutes au lieu de semaines. |

#### 1.1.2 Les trois modèles de service — *IaaS / PaaS / SaaS*

| Modèle | Ce que **vous** gérez | Ce que **le cloud** gère | Exemple Azure |
|--------|----------------------|--------------------------|---------------|
| **IaaS** (Infrastructure as a Service) | OS, runtimes, code, données | Hyperviseur, matériel, réseau | Azure **Virtual Machines** |
| **PaaS** (Platform as a Service) | Code, données, configuration | OS, runtime, patchs, scaling | Azure **App Service**, **Functions** |
| **SaaS** (Software as a Service) | Données et utilisateurs | Tout le reste | **Microsoft 365**, **Dynamics** |

C'est le **modèle de responsabilité partagée** : plus on monte de IaaS vers SaaS, moins on a de choses à gérer — mais aussi moins de contrôle.

👉 Dans ce notebook, on se concentre sur le **PaaS** : on déploie du code sans se soucier des VMs.

#### 1.1.3 Les modèles de déploiement

- **Public cloud** — partagé, hébergé par Microsoft (le cas par défaut, et celui qu'on utilise).
- **Privé** — chez vous ou un hébergeur dédié.
- **Hybride** — un mix des deux, très courant en entreprise.
- **Multi-cloud** — plusieurs fournisseurs combinés (Azure + AWS + GCP).

#### 1.1.4 L'architecture physique d'Azure

| Concept | Description |
|---------|-------------|
| **Région** | Une zone géographique avec ≥ 1 datacenter (ex. `francecentral`, `westeurope`, `eastus`). |
| **Availability Zone (AZ)** | Datacenters physiquement séparés *à l'intérieur d'une région*. Multi-AZ = haute dispo. |
| **Region pair** | Chaque région a un binôme pour la reprise après sinistre (ex. France Central ↔ France South). |
| **Sovereign region** | Régions souveraines (Azure Government, Azure China) — isolation totale. |

#### 1.1.5 L'organisation logique (hiérarchie Azure)

```
Tenant Microsoft Entra ID
 └── Management Group       (regroupe plusieurs abonnements)
      └── Subscription      (l'unité de facturation)
           └── Resource Group (dossier logique de ressources)
                └── Resources (App Service, BDD, etc.)
```

- **Subscription** : où sont facturées les ressources.
- **Resource Group (RG)** : un conteneur logique. **Toutes** vos ressources doivent appartenir à un RG. Supprimer un RG supprime **tout** ce qu'il contient — pratique pour le nettoyage, dangereux si on se trompe.
- **SKU** (Stock Keeping Unit) : la « taille »/tier d'un service (ex. `F1` gratuit, `B1` basic, `S1` standard…).
- **Azure Resource Manager (ARM)** : l'API unifiée qui orchestre tout. L'`az cli`, le portail, Bicep, Terraform passent tous par ARM.

#### 1.1.6 Les grandes familles de services Azure

| Famille | Services-phares | À quoi ça sert |
|---------|-----------------|----------------|
| **Compute** | VMs, Scale Sets, **App Service**, Functions, Container Apps, AKS | Exécuter du code |
| **Storage** | Blob, Files, Queues, Tables, Disks | Stocker des données |
| **Networking** | VNet, Load Balancer, App Gateway, VPN Gateway, ExpressRoute, DNS | Connecter, sécuriser, équilibrer |
| **Databases** | Azure SQL, Cosmos DB, PostgreSQL flexible, Redis | Relationnel, NoSQL, cache |
| **Identity** | **Microsoft Entra ID**, RBAC, MFA, Conditional Access | Qui peut faire quoi sur quoi |
| **IA** | **Microsoft Foundry**, AI Search, Document Intelligence, Speech | Modèles, agents, vision, RAG, voix |
| **Observabilité** | Azure Monitor, **Application Insights**, Log Analytics | Logs, métriques, traces |
| **Governance** | Policy, Cost Management, Tags, Blueprints | Maîtriser dépense & conformité |

#### 1.1.7 Identité, sécurité & gouvernance (l'essentiel)

- **Microsoft Entra ID** = l'annuaire d'identités Microsoft (ex-Azure AD). C'est lui qui sait qui vous êtes.
- **RBAC** (Role-Based Access Control) : on assigne des **rôles** (`Owner`, `Contributor`, `Reader`, custom…) à des **identités** (utilisateurs, groupes, applications) sur des **scopes** (subscription, RG, ressource individuelle).
- **Managed Identity** : permet à une appli Azure (ex. App Service) d'appeler une autre ressource (ex. Foundry) **sans clé à gérer**. Couvert dans le notebook 05.
- **Azure Policy** : règles à respecter (ex. *« interdit de créer une VM en dehors de l'Europe »*).
- **Tags** : étiquettes `clé=valeur` posées sur les ressources, utiles pour la facturation et le tri.

> 💰 **Pricing** : tout est facturé à l'usage. Azure fournit un [calculateur de prix](https://azure.microsoft.com/pricing/calculator/) et un service **Cost Management** pour suivre la dépense. Pour ce parcours, on choisit volontairement des SKU gratuits (`F1`) ou très bas coût.

### 1.2 IA — du ML à la GenAI — *style AI-900*

#### 1.2.1 Vue d'ensemble — IA ⊃ ML ⊃ DL ⊃ GenAI

Ces termes sont **emboîtés** comme des poupées russes :

```
┌────────────────────────────────────────────────────────────┐
│ Intelligence Artificielle (IA)                             │
│  toute approche qui imite des capacités humaines           │
│  (logique, perception, raisonnement, langage…)             │
│                                                            │
│  ┌──────────────────────────────────────────────────────┐  │
│  │ Machine Learning (ML)                                │  │
│  │  apprendre à partir de données, sans être           │  │
│  │  programmé règle par règle                           │  │
│  │                                                      │  │
│  │  ┌────────────────────────────────────────────────┐  │  │
│  │  │ Deep Learning (DL)                             │  │  │
│  │  │  ML avec des réseaux de neurones profonds      │  │  │
│  │  │                                                │  │  │
│  │  │  ┌──────────────────────────────────────────┐  │  │  │
│  │  │  │ IA générative (GenAI)                    │  │  │  │
│  │  │  │  DL qui *génère* (texte, image, code…)   │  │  │  │
│  │  │  │  → LLM, diffusion models, etc.           │  │  │  │
│  │  │  └──────────────────────────────────────────┘  │  │  │
│  │  └────────────────────────────────────────────────┘  │  │
│  └──────────────────────────────────────────────────────┘  │
└────────────────────────────────────────────────────────────┘
```

Retenez : **toute GenAI est du Deep Learning, mais tout le ML n'est pas du Deep Learning**. Et on a fait de l'IA bien avant le ML (systèmes experts, règles, recherche de chemin…).

---

#### 1.2.2 Les grandes tâches d'IA — *workloads AI-900*

L'AI-900 classe les usages de l'IA en grandes familles. Connaître ce catalogue aide à choisir le bon service Azure pour le bon problème :

| Type de tâche | Question répondue | Exemple | Service Azure typique |
|---------------|-------------------|---------|----------------------|
| **Prediction / Forecasting** | Quelle valeur future ? | Prévision de ventes | Azure ML, AutoML |
| **Classification** | Est-ce A, B ou C ? | Spam / non-spam | Azure ML, Foundry |
| **Anomaly detection** | Est-ce anormal ? | Détection de fraude | Azure ML, Anomaly Detector |
| **Computer vision** | Que voit-on sur l'image ? | OCR, détection d'objets, reconnaissance faciale | **Azure AI Vision** |
| **Natural Language Processing (NLP)** | De quoi parle ce texte ? | Sentiment, traduction, NER, résumé | **Azure AI Language**, **Speech** |
| **Document intelligence** | Structurer un document | Extraction de tableaux et formulaires de PDF | **Azure AI Document Intelligence** |
| **Knowledge mining** | Retrouver l'info | Recherche sémantique, **RAG** | **Azure AI Search**, Foundry IQ |
| **Generative AI** | Créer du contenu | Texte, image, code, audio, vidéo | **Microsoft Foundry** |

> 🎯 Dans ce parcours on se focalise sur la **GenAI**, mais sachez que les autres familles existent et sont souvent **complémentaires** (ex. un agent qui appelle Document Intelligence pour parser un PDF avant de répondre).

---

#### 1.2.3 Machine Learning — les fondations

Un modèle de ML apprend une fonction `f(X) → y` à partir d'exemples. On distingue **trois grandes familles** d'apprentissage :

| Type | Données | Exemple concret | Algos typiques |
|------|---------|-----------------|----------------|
| **Supervisé** | Données + **labels** (la réponse attendue) | Prédire le prix d'un appart à partir de sa surface | Régression linéaire, régression logistique, arbres de décision, Random Forest, Gradient Boosting (XGBoost), SVM |
| **Non supervisé** | Données seules, pas de labels | Segmenter une base clients en groupes similaires | K-means, DBSCAN, ACP/PCA, autoencodeurs |
| **Par renforcement** | Un agent qui interagit avec un environnement et reçoit des récompenses | Apprendre à jouer aux échecs, piloter un robot | Q-learning, PPO, RLHF (utilisé pour affiner ChatGPT) |

##### Les deux grandes tâches en supervisé

- **Régression** → prédire une valeur **continue** (ex. prix, température, durée).  
  Exemple : *« combien va coûter cet appartement ? »* → 285 000 €.  
  Algo le plus simple : la **régression linéaire** `y = a·x + b`.

- **Classification** → prédire une **catégorie**.  
  Exemple : *« cet email est-il un spam ? »* → oui / non.  
  Algos : régression **logistique** (malgré son nom, c'est de la classif), arbres, SVM…

##### Le cycle d'un projet ML

1. **Collecter** des données.
2. **Préparer** : nettoyer, normaliser, *feature engineering*.
3. **Entraîner** : on découpe en jeux *train / validation / test*. Le modèle ajuste ses **paramètres** pour minimiser une *fonction de perte* (loss).
4. **Évaluer** : précision, rappel, F1, RMSE… selon la tâche.
5. **Déployer** et **monitorer** la dérive (le monde change, le modèle vieillit).

> 🧠 Vocabulaire-clé : *features* (entrées), *labels* (sorties attendues), *overfitting* (le modèle apprend par cœur le train et rate le test), *hyperparamètres* (réglages choisis avant l'entraînement, ex. nombre d'arbres).

---

#### 1.2.4 Deep Learning & réseaux de neurones

Un **neurone artificiel** fait une opération très simple : il prend des entrées, les pondère, additionne, et passe le résultat dans une fonction non-linéaire (*activation*, ex. `ReLU`).

```
x1 ─┐
x2 ─┼─► [Σ wi·xi + b] ─► activation ─► sortie
x3 ─┘
```

Empilez des milliers de ces neurones en **couches** → vous obtenez un **réseau de neurones profond** (deep = beaucoup de couches). L'entraînement utilise la **rétropropagation du gradient** (*backprop*) : on calcule à quel point chaque poids a contribué à l'erreur, et on l'ajuste un peu dans le bon sens. Répétez des millions de fois.

##### Pourquoi le DL a explosé vers 2012 ?

Trois facteurs simultanés :
1. **Données massives** (Internet, ImageNet, etc.).
2. **GPU** capables de paralléliser les calculs matriciels.
3. **Algorithmes** matures (ReLU, dropout, batch normalization…).

##### Architectures classiques à connaître

| Archi | Bonne pour | Exemple |
|------|------------|---------|
| **CNN** (Convolutional Neural Network) | Images | Détection d'objets, vision médicale |
| **RNN / LSTM / GRU** | Séquences (texte, séries temporelles) | Traduction d'avant 2017, prévision météo |
| **Transformer** ⭐ | Séquences, mais en **parallèle** grâce à l'**attention** | GPT, BERT, Llama, Mistral, Phi… |

Le **Transformer** (article *« Attention is All You Need »*, Google 2017) est l'innovation qui a rendu possible l'IA générative moderne. Son mécanisme d'**attention** permet à chaque mot d'« regarder » tous les autres mots de la séquence pour comprendre le contexte — et tout ça en parallèle, ce qui scale très bien sur GPU.

---

#### 1.2.5 L'arrivée de l'IA générative

L'**IA générative** est un Deep Learning dont la sortie n'est plus une classe ou un nombre, mais **du contenu** (texte, code, image, audio, vidéo, 3D…).

##### Familles de modèles génératifs

| Sortie | Famille | Exemples |
|--------|---------|----------|
| Texte / code | **LLM** (Large Language Model), basés sur Transformer | GPT-5, Claude, Llama 3, Mistral, Phi |
| Image | **Diffusion models** | DALL·E, Stable Diffusion, FLUX |
| Audio / parole | TTS, Whisper-like | Azure Speech, ElevenLabs |
| Vidéo | Diffusion vidéo | Sora |
| Multimodal | Modèles « omni » | GPT-4o, Gemini |

##### Comment un LLM fonctionne (en 1 minute)

1. On lui donne un **prompt** (du texte).
2. Il prédit **le prochain token** le plus probable.
3. Il rajoute ce token au prompt, et recommence.
4. Il s'arrête quand il génère un token spécial *fin* ou atteint la limite.

C'est tout. Un LLM est, fondamentalement, **une machine très sophistiquée à prédire le token suivant**.

---

#### 1.2.6 Concepts essentiels de la GenAI

##### Token

Un **token** ≈ un morceau de mot. En anglais, **~4 caractères** ou **~0,75 mot** par token. En français, c'est un peu plus coûteux.

| Texte | ~ Tokens |
|-------|---------:|
| `Hello world` | 2 |
| `Bonjour le monde` | 4 |
| `import pandas as pd` | 5 |

> 💰 **La facturation des LLM se fait au token** (input + output). Et la **context window** (mémoire courante du modèle) se mesure aussi en tokens. Compter ses tokens, c'est compter ses sous.

##### Context window

C'est la **quantité de tokens** que le modèle peut « voir » en une fois (prompt + historique + sortie).
- GPT-4o : 128k tokens
- GPT-4.1 / GPT-5 : jusqu'à 1M+
- Claude Sonnet : 200k+
- Llama 3 : 8k → 128k selon variante

Plus la fenêtre est grande, plus on peut y caser de contexte (documents, historique de conversation) — mais aussi plus c'est cher et lent.

##### LLM vs SLM

| | **LLM** (Large) | **SLM** (Small) |
|---|---|---|
| Taille (paramètres) | 70B → 1T+ | < 10B (souvent 1–7B) |
| Hébergement | Cloud, GPU coûteux | Cloud léger, edge, voire **on-device** (téléphone, navigateur) |
| Latence | Élevée | Faible |
| Coût | Élevé | Très bas |
| Cas d'usage | Tâches complexes, raisonnement, créatif | Tâches ciblées, extraction, classification, on-device |
| Exemples | GPT-5, Claude Opus, Llama 70B | **Phi-4**, Llama 3.2 1B/3B, Mistral 7B, Gemma 2 |

> 🎯 La tendance 2025-2026 : combiner **un SLM rapide en première ligne** + un **LLM puissant en escalade** quand c'est nécessaire. C'est plus rapide et bien moins cher.

##### Embeddings

Un **embedding**, c'est un **vecteur** (liste de nombres, ex. 1536 dimensions) qui représente le **sens** d'un texte. Deux textes proches sémantiquement ont des vecteurs proches (distance cosine faible).

Usage typique : la **recherche sémantique** et le **RAG** (cf. ci-dessous).

##### Training vs Fine-tuning vs Inference

- **Pre-training** : entraîner un modèle de zéro sur des téraoctets de texte. Coûte des **millions de dollars**. Quelques acteurs au monde.
- **Fine-tuning** : ré-entraîner légèrement un modèle pré-entraîné sur vos données pour le spécialiser. Beaucoup moins cher.
- **Inference** : *utiliser* le modèle (lui poser une question). C'est ce qu'on fait 99 % du temps.

##### Prompt, system prompt, RAG, agents — la pyramide

1. **Prompt** : ce qu'on écrit au modèle.
2. **System prompt** : instructions de cadrage (« tu es un assistant comptable… »).
3. **Few-shot** : on glisse 2-3 exemples dans le prompt pour guider la réponse.
4. **RAG** (Retrieval-Augmented Generation) : avant de répondre, on **cherche** dans une base de documents les passages pertinents, et on les injecte dans le prompt. → Le modèle répond avec des **sources** et sans hallucinations.
5. **Agent** : un LLM + des **outils** (fonctions à appeler) + une **boucle** de raisonnement. L'agent décide quoi faire, appelle des outils, lit les résultats, et continue jusqu'à atteindre l'objectif.

##### Hallucinations

Un LLM peut **inventer** des faits avec aplomb. C'est inhérent au mécanisme. Les parades : RAG, *grounding*, citation des sources, évaluation systématique, et un humain dans la boucle pour les sujets sensibles.

---

#### 1.2.7 Responsible AI — *les 6 principes Microsoft*

L'AI-900 insiste beaucoup sur ce point. Microsoft a publié **6 principes** que tout système d'IA doit respecter :

| Principe | Question à se poser |
|----------|---------------------|
| **Fairness** (équité) | Mon modèle traite-t-il toutes les populations équitablement ? Pas de biais genre, ethnie, âge ? |
| **Reliability & safety** | Se comporte-t-il de façon prévisible, même dans les cas limites ? |
| **Privacy & security** | Les données personnelles sont-elles protégées ? Pas de fuite via les prompts ? |
| **Inclusiveness** | Accessible à tous (handicaps, langues, contextes culturels) ? |
| **Transparency** | Peut-on expliquer pourquoi l'IA a pris telle décision ? |
| **Accountability** | Y a-t-il un humain responsable in fine ? |

> 🛡️ Concrètement chez Microsoft : **Azure AI Content Safety**, filtres intégrés à Foundry, *system messages* contraignants, jeux d'**évaluations** systématiques (cf. notebook 06).

---

> ✅ **À ce stade vous savez :** ce qu'est l'IA, le ML, le DL, la GenAI ; les algos de régression / classification ; les grandes tâches d'IA et les services Azure associés ; comment marche un réseau de neurones ; ce qu'est un LLM, un SLM, un token, une context window, un embedding ; le vocabulaire prompt / RAG / agent ; les 6 principes Responsible AI.
>
> Place à **la plateforme qui regroupe tout ça côté Microsoft → chapitre 2 : Microsoft Foundry.**

## 2. Microsoft Foundry — la plateforme IA d'Azure

Maintenant qu'on a les concepts (1.2), **où fait-on tout ça concrètement chez Microsoft ?** Réponse : **Microsoft Foundry** (anciennement *Azure AI Foundry / Azure AI Studio*).

Foundry, c'est **la plateforme unifiée** pour bâtir et opérer des applications IA — du prototype à la production.

### 2.1 Ce qu'on y trouve

| Brique Foundry | À quoi ça correspond dans le 1.2 |
|----------------|----------------------------------|
| **Model catalog** | Le supermarché des modèles : GPT-5, Llama, Mistral, **Phi (SLM Microsoft)**, DeepSeek, modèles d'images, de speech… |
| **Model deployments** | L'**inference** d'un modèle exposé en endpoint HTTPS (pay-per-token). |
| **Playgrounds** | Tester un prompt / une conversation sans écrire une ligne de code. |
| **Agents** | Construire des **agents** (LLM + outils + boucle) en quelques clics ou en SDK. |
| **Indexes** | Bases vectorielles d'**embeddings** → support du **RAG**. |
| **Datasets** | Jeux de données pour évaluer ou *fine-tuner*. |
| **Evaluations** | Mesurer la qualité d'un modèle ou d'un agent (groundedness, relevance, safety…). |
| **Tracing** | Voir ce que fait l'agent étape par étape — exporté vers **Application Insights** (branché dans le notebook 06). |
| **Content safety** | Filtres pour bloquer prompts malveillants et sorties à risque — bras armé du *Responsible AI*. |
| **Fine-tuning** | Spécialiser un modèle sur vos données. |

### 2.2 Hiérarchie des ressources

```
Foundry resource  (Azure resource, kind = AIServices, project management activé)
 │
 ├── Foundry project #1   (« default »)
 │    ├── model deployments (gpt-5-mini, text-embedding-3-large, …)
 │    ├── agents
 │    ├── indexes / datasets
 │    └── evaluations / traces
 │
 └── Foundry project #2   (équipe / produit différent)
      └── …
```

- **Une seule ressource Foundry** peut héberger **plusieurs projets** — pratique pour isoler des équipes tout en mutualisant les modèles déployés et la sécurité.
- Les **projets** sont des sous-ressources Azure, ils ont leur propre RBAC et leur propre identité.
- L'authentification se fait via **Microsoft Entra ID** (`DefaultAzureCredential` côté SDK). Pas de clé API à se trimballer.

### 2.3 Pourquoi Foundry plutôt que d'appeler l'API OpenAI en direct ?

- **Sécurité & compliance Azure** : tenant Entra ID, RBAC, private endpoints, content safety, journaux d'audit.
- **Catalogue multi-modèles** : on n'est pas verrouillé sur un fournisseur. Llama, Mistral, Phi, GPT… au même endroit, même API d'inférence.
- **Intégration native** avec App Service, Application Insights, Key Vault, Cosmos DB, AI Search…
- **Tooling fini** : évaluations, traces, content safety, fine-tuning — au lieu de tout réécrire à la main.

### 2.4 Ce qu'on va faire dans la suite du notebook

Plus loin (chapitre 7), on va **provisionner avec l'`az cli`** :
1. Une **ressource Foundry** (`kind = AIServices`, project management activé).
2. Un **custom subdomain** (requis pour l'authentification Entra ID).
3. Un **projet** dans cette ressource.

Puis on ouvrira https://ai.azure.com pour visualiser tout ça. Le **déploiement d'un modèle et la création d'un premier agent** seront couverts dans le **notebook 02**, et le **monitoring + évaluation** dans le **notebook 06**.

📚 Pour aller plus loin : https://learn.microsoft.com/azure/foundry/

## 3. Pré-requis : Azure CLI & SDK Python

On vérifie que l'`az cli` est installée (version recommandée **≥ 2.86**). Si la commande ci-dessous échoue ou affiche une version plus ancienne, mettez-la à jour : https://learn.microsoft.com/cli/azure/install-azure-cli

- Mettre à jour l'`az cli` en place : `az upgrade`
- Lister les extensions installées : `az extension list -o table`

In [ ]:
!az --version

### SDK Python Azure (dernières versions stables)

On installe les SDK Python utilisés dans ce notebook et les suivants. Les versions sont **épinglées** aux dernières stables vérifiées sur PyPI (mai 2026). Un `requirements.txt` est aussi présent à la racine du repo si vous préférez `pip install -r ../requirements.txt`.

| Package | Rôle | Version min |
|---------|------|------------:|
| `azure-identity` | Authentification (DefaultAzureCredential, MSI…) | `1.25.3` |
| `azure-mgmt-cognitiveservices` | Gestion (CRUD) des ressources Foundry / Cognitive Services | `14.1.0` |
| `azure-ai-projects` | SDK plan **data** pour les projets Foundry | `2.1.0` |
| `azure-ai-agents` | SDK pour créer/invoquer des agents Foundry | `1.1.0` |
| `azure-monitor-opentelemetry` | Tracing + logs vers Application Insights via OpenTelemetry | `1.8.8` |

In [ ]:
%pip install --upgrade \
    "azure-identity>=1.25.3" \
    "azure-mgmt-cognitiveservices>=14.1.0" \
    "azure-ai-projects>=2.1.0" \
    "azure-ai-agents>=1.1.0" \
    "azure-monitor-opentelemetry>=1.8.8"

### Connexion à Azure

La commande suivante ouvre une fenêtre de navigateur pour vous authentifier. Si vous êtes sur une machine sans navigateur, utilisez `az login --use-device-code`.

In [ ]:
!az login

### Lister et choisir votre abonnement

Si vous avez plusieurs abonnements, sélectionnez celui sur lequel vous voulez travailler.

In [ ]:
!az account list --output table

In [ ]:
# 👉 Remplacez par le nom ou l'ID de votre abonnement, puis exécutez.
SUBSCRIPTION = "<votre-subscription-name-ou-id>"
!az account set --subscription "{SUBSCRIPTION}"
!az account show --output table

## 4. Définir nos variables

Votre équipe vous a **créé un Resource Group** au nom suivant la convention :

```
rg-stage-<votre-identifiant>
```

…et vous a accordé le rôle **Owner** (RBAC) dessus. Vous pouvez donc y créer/supprimer librement des ressources, **sans toucher au RG lui-même**.

👉 **Action requise** : remplacez `<votre-identifiant>` ci-dessous par votre identifiant (ex. `leo`, `martin42`…). C'est la **seule** variable à éditer dans ce notebook — toutes les autres en découlent :

- La région (`LOCATION`) est lue automatiquement depuis le RG.
- Les noms de ressources enfants suivent la convention `*-stage-<votre-identifiant>` (App Service Plan, Web App, Foundry…).
- Vous n'avez **rien à recopier** entre notebooks : 02 et 06 demanderont juste votre RG et déduiront tout le reste.

In [ ]:
import os, re, subprocess

# 👇 SEULE VARIABLE À ÉDITER : le nom de VOTRE Resource Group.
#    Convention équipe : rg-stage-<votre-identifiant>
RG = "rg-stage-<votre-identifiant>"

# Extraction automatique de l'identifiant utilisateur depuis le nom du RG.
m = re.match(r"^rg-stage-(?P<id>[a-z0-9\-]+)$", RG)
assert m, (
    f"❌ Le RG '{RG}' ne suit pas la convention attendue 'rg-stage-<id>'. "
    "Renseignez-le exactement comme on vous l'a fourni."
)
USER_ID = m.group("id")

# Région lue automatiquement depuis le RG (vérifie aussi qu'il existe et qu'on a accès)
LOCATION = subprocess.check_output(
    f"az group show --name {RG} --query location -o tsv", shell=True
).decode().strip()
assert LOCATION, (
    "❌ Impossible de lire la région du RG. Vérifiez son nom exact et "
    "que vous avez bien le rôle Owner/Contributor dessus."
)

# Noms de ressources alignés sur la convention -stage-<id>
PLAN     = f"asp-stage-{USER_ID}"   # App Service Plan
WEBAPP   = f"web-stage-{USER_ID}"   # Web App (DNS public unique)
FOUNDRY  = f"aif-stage-{USER_ID}"   # Ressource Foundry (Cognitive Services AIServices)
PROJECT  = "my-foundry-project"     # Projet Foundry (interne au compte Foundry)

# Exporte vers l'environnement pour les cellules shell (!az ...)
for k, v in {
    "LOCATION": LOCATION, "RG": RG, "USER_ID": USER_ID, "PLAN": PLAN,
    "WEBAPP": WEBAPP, "FOUNDRY": FOUNDRY, "PROJECT": PROJECT,
}.items():
    os.environ[k] = v

print(f"Resource Group : {RG}  (région détectée : {LOCATION})")
print(f"Identifiant    : {USER_ID}")
print(f"Web App        : https://{WEBAPP}.azurewebsites.net")
print(f"Foundry        : {FOUNDRY}")
print("\n✅ Tout est dérivé du RG. Pour les notebooks 02 et 06, vous n'aurez qu'à coller le même nom de RG.")

## 5. Vérifier le Resource Group existant

Vous **n'allez pas créer** de Resource Group : votre équipe vous en a déjà fourni un, avec un rôle **Owner** ou **Contributor** dessus. On vérifie juste qu'il existe bien et qu'on y a accès.

In [ ]:
# Vérification : le RG existe, on a un rôle dessus, et il est dans la bonne région.
!az group show --name $RG --output table

# Lister mes rôles sur ce RG (doit contenir Owner ou Contributor) — facultatif mais rassurant.
!az role assignment list --scope $(az group show --name $RG --query id -o tsv) --assignee $(az ad signed-in-user show --query id -o tsv) --query "[].roleDefinitionName" -o tsv 2>nul

## 6. App Service — héberger une application web

**App Service** est le PaaS d'hébergement web d'Azure. On fournit du code (ou un conteneur), Azure s'occupe de l'OS, du runtime, des patchs, du HTTPS, du scaling.

Deux objets à créer :

1. **App Service Plan** — la « machine » sous-jacente (CPU/RAM). On prend `F1` (gratuit, Linux).
2. **Web App** — l'application qui tourne sur ce plan.

> Le plan `F1` a des limites : 1 Go RAM, 60 min de CPU/jour, pas d'auto-scale. Parfait pour apprendre.

In [ ]:
# 6.1 Créer l'App Service Plan (Linux, SKU gratuit F1)
!az appservice plan create \
    --name $PLAN \
    --resource-group $RG \
    --sku F1 \
    --is-linux \
    --output table

In [ ]:
# 6.2 Créer la Web App avec un runtime Python 3.13 (dernière stable largement supportée par l'écosystème)
#     Pour voir la liste complète : !az webapp list-runtimes --os linux -o tsv | findstr PYTHON
!az webapp create \
    --name $WEBAPP \
    --resource-group $RG \
    --plan $PLAN \
    --runtime "PYTHON:3.13" \
    --output table

In [ ]:
# 6.3 Récupérer l'URL publique de l'app
!az webapp show --name $WEBAPP --resource-group $RG --query defaultHostName -o tsv

Ouvrez l'URL dans votre navigateur. Vous devriez voir la page d'accueil par défaut « Your web app is running and waiting for your content ». Pour déployer du code par la suite, on utiliserait `az webapp up`, un déploiement Git, ou un pipeline Azure DevOps / GitHub Actions.

## 7. Provisionner Microsoft Foundry

On a vu en **chapitre 2** ce qu'est Foundry et son modèle de ressources. On passe maintenant à la pratique : créer la ressource, son sous-domaine personnalisé, et un projet, avec l'`az cli`.

Trois étapes :

1. **Créer la ressource Foundry** (`kind = AIServices`, project management activé).
2. **Configurer un custom subdomain** — requis pour l'authentification Entra ID.
3. **Créer un projet** dans cette ressource.

📚 Doc : https://learn.microsoft.com/azure/foundry/how-to/create-projects

In [ ]:
# 7.1 Créer la ressource Foundry (Cognitive Services kind=AIServices)
#     --allow-project-management active la possibilité de créer des projets dessus.
!az cognitiveservices account create \
    --name $FOUNDRY \
    --resource-group $RG \
    --kind AIServices \
    --sku S0 \
    --location $LOCATION \
    --allow-project-management true \
    --yes \
    --output table

In [ ]:
# 7.2 Attribuer un custom sub-domain (requis pour les appels Foundry, doit être unique mondialement)
!az cognitiveservices account update \
    --name $FOUNDRY \
    --resource-group $RG \
    --custom-domain $FOUNDRY \
    --output table

In [ ]:
# 7.3 Créer un projet Foundry dans la ressource
!az cognitiveservices account project create \
    --name $FOUNDRY \
    --resource-group $RG \
    --project-name $PROJECT \
    --location $LOCATION \
    --output table

In [ ]:
# 7.4 Vérifier le projet
!az cognitiveservices account project show \
    --name $FOUNDRY \
    --resource-group $RG \
    --project-name $PROJECT \
    --output jsonc

🎉 Ouvrez ensuite **https://ai.azure.com** : vous y retrouverez votre projet `my-foundry-project`. Dans les prochains notebooks, on y déploiera un modèle et on y créera un premier agent.

## 8. Vue d'ensemble de ce qu'on a créé

Listons toutes les ressources du Resource Group pour visualiser ce qui a été déployé.

In [ ]:
!az resource list --resource-group $RG --output table

## 9. 🧹 Nettoyage — IMPORTANT

Le **Resource Group ne vous appartient pas** : il a été créé par votre équipe et peut contenir d'autres ressources. **Ne le supprimez surtout pas** (`az group delete` est **interdit ici**).

À la place, on supprime **uniquement les ressources créées par ce notebook**, une par une.

> ⚠️ Une fois supprimées, ces ressources sont **irrécupérables**.
>
> 💡 Si vous comptez enchaîner sur les notebooks **02 (Foundry agent)** ou **06 (monitoring + évaluation)**, **gardez** tout en place — ils réutilisent les ressources créées ici.

In [ ]:
# Nettoyage ciblé : supprimer UNIQUEMENT les ressources créées par ce notebook.
# Décommentez les lignes que vous voulez exécuter.

# --- Foundry (projet, puis ressource) ---
# !az cognitiveservices account project delete --name $FOUNDRY --resource-group $RG --project-name $PROJECT
# !az cognitiveservices account delete         --name $FOUNDRY --resource-group $RG

# --- App Service (Web App, puis plan) ---
# !az webapp delete         --name $WEBAPP --resource-group $RG
# !az appservice plan delete --name $PLAN   --resource-group $RG --yes

print("⚠️  Ne supprimez PAS le Resource Group : il appartient à votre équipe.")
print(f"    RG protégé : {RG}")
print("    Décommentez ci-dessus uniquement les ressources que VOUS avez créées.")

## Récap & prochaines étapes

Vous savez maintenant :
- **§ 1.1 Azure** : ce qu'est le **cloud**, IaaS / PaaS / SaaS, hiérarchie Azure, régions, AZ, SKU, RBAC, services-phares ;
- **§ 1.2 IA** : ML / DL / GenAI, tâches d'IA AI-900, LLM/SLM, tokens, embeddings, RAG, agents, Responsible AI ;
- **§ 2 Foundry** : ce qu'est Microsoft Foundry et comment il s'inscrit dans l'écosystème ;
- vous connecter avec `az login` et choisir un abonnement ;
- réutiliser le **Resource Group** fourni par votre équipe (sans le supprimer) ;
- déployer une **Web App** sur **App Service** ;
- provisionner une ressource et un projet **Microsoft Foundry** ;
- nettoyer proprement (ressource par ressource, jamais le RG).

**Prochains notebooks :**
1. **02 — Déployer un modèle + un Agent IA + un Workflow dans Foundry** : la suite logique pratique.
2. **03 — Projet, DevOps, Agile & collaboration** : comment ça se passe en équipe (sprints, PR, CI/CD…).
3. **04 — Architecture web** : API, frontend, backend, comment les pièces s'assemblent.
4. **05 — Sécurité cloud** : Entra ID, RBAC, Managed Identity, Key Vault (survol).
5. **06 — Monitoring + évaluation** : Application Insights, KQL, tracing agents et qualité.

📚 Pour aller plus loin :
- Parcours **AZ-900** *Azure Fundamentals* : https://learn.microsoft.com/training/paths/microsoft-azure-fundamentals-describe-cloud-concepts/
- Parcours **AI-900** *AI Fundamentals* : https://learn.microsoft.com/training/paths/get-started-with-artificial-intelligence-on-azure/
- Doc App Service : https://learn.microsoft.com/azure/app-service/
- Doc Microsoft Foundry : https://learn.microsoft.com/azure/foundry/
- Principes **Responsible AI** Microsoft : https://www.microsoft.com/ai/responsible-ai
- *Attention is All You Need* (l'article fondateur du Transformer) : https://arxiv.org/abs/1706.03762